In [12]:
from pyarrow import flight
import time

user = "kamlesh.sharma+demo@dremio.com"
password = "dremio123"
sql = 'SELECT * from INFORMATION_SCHEMA."tables"'

def connect_to_dremio_flight_server_endpoint(hostname, flightport, username, password, sqlquery):
    try:
        # Default to use an unencrypted TCP connection.
        scheme = "grpc+tcp"
        connection_args = {}
 
        client = flight.FlightClient("{}://{}:{}".format(scheme, hostname, flightport),
          middleware=[], **connection_args)

        # Authenticate with the server endpoint.
        bearer_token = client.authenticate_basic_token(username, password)
        print('[INFO] Authentication was successful')

        if sqlquery:
            # Construct FlightDescriptor for the query result set.
            flight_desc = flight.FlightDescriptor.for_command(sqlquery)
            print('[INFO] Query: ', sqlquery)

            # Retrieve the schema of the result set.
            options = flight.FlightCallOptions(headers=[bearer_token])

            # Get the FlightInfo message to retrieve the Ticket corresponding to the query result set.
            flight_info = client.get_flight_info(flight_desc, options)

            # Retrieve the result set as a stream of Arrow record batches.
            reader = client.do_get(flight_info.endpoints[0].ticket, options)
            print(reader.read_pandas())

    except Exception as exception:
        print("[ERROR] Exception: {}".format(repr(exception)))
        raise

def flight_query():
    start = time.time()
    connect_to_dremio_flight_server_endpoint("dremio.org",32010,user,password,sql)
    elapsed = time.time() - start
    return elapsed

flight_result = flight_query()
print('[INFO] Arrow Flight: ', flight_result)

[INFO] Authentication was successful
[INFO] Query:  SELECT * from INFORMATION_SCHEMA."tables"
    TABLE_CATALOG            TABLE_SCHEMA              TABLE_NAME  \
0          DREMIO        Demo.Application  nyc_trips_with_weather   
1          DREMIO        Demo.Application     refinery_monitoring   
2          DREMIO        Demo.Application       store_sales_tpcds   
3          DREMIO  Demo.Business.Customer        customer_address   
4          DREMIO     Demo.Business.Store                    item   
..            ...                     ...                     ...   
821        DREMIO                     sys                 version   
822        DREMIO                     sys                   views   
823        DREMIO           unity.default               nyc_trips   
824        DREMIO           unity.default        trips_pickupdate   
825        DREMIO   wolfgang.socar-bucket            META.parquet   

       TABLE_TYPE  
0            VIEW  
1            VIEW  
2            VIEW